# H-007 · GDELT data-vendors probe

Vendor probe for **H-007 Sentiment** using GDELT 2.0 Global Knowledge Graph (`V2Tone`) against the S1 equities train panel.

**Questions this notebook answers**

1. How far back does usable GDELT `V2Tone` coverage go for our names?
2. How do we map panel tickers → GDELT organization strings — and will mapping be a problem?
3. How do we access the same data daily for forward testing / live trading?

**Locked research decisions**

- Train and serve on the **same** source (GDELT end-to-end).
- Start with feed `V2Tone`; consider FinBERT later only if IC is weak.
- History via BigQuery `gdelt-bq.gdeltv2.gkg_partitioned`; live via HTTP 15-minute GKG files.

**Non-goals**

- No IC / Alphalens, no FinBERT, no production fetcher, no `feature_store` entrypoint.

**Prerequisites**

- GCP project with BigQuery API enabled and Application Default Credentials (`gcloud auth application-default login`).
- Python packages: `google-cloud-bigquery`, `pandas`, `requests`, `matplotlib`.
- Network access to SEC (`company_tickers.json`) and `data.gdeltproject.org`.


## 0. Imports & Config

Paths, probe knobs, and panel ticker load. Helpers for SEC aliases, `V2Tone` parsing, and BigQuery / HTTP access live here so §1–§3 reuse the same mapping logic.


In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
import zipfile
from collections import defaultdict
from typing import Iterable

import matplotlib.pyplot as plt
import pandas as pd
import requests

# Optional until §1 / §2 BigQuery cells run
try:
    from google.cloud import bigquery
except ImportError:  # pragma: no cover
    bigquery = None
    print("google-cloud-bigquery not installed - install before running §1/§2")

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
CACHE_DIR = os.path.join(ROOT, "01_data", "cache", "alternative_data", "gdelt_probe")
os.makedirs(CACHE_DIR, exist_ok=True)

SEC_TICKER_MAP_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_USER_AGENT = os.environ.get(
    "SEC_USER_AGENT", "trading_portfolio charlie.vellacott@gmail.com"
)
GDELT_LASTUPDATE_URL = "http://data.gdeltproject.org/gdeltv2/lastupdate.txt"
BQ_GKG_TABLE = "`gdelt-bq.gdeltv2.gkg_partitioned`"

# Focus names for expensive BigQuery tone / coverage pulls
FOCUS_TICKERS = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META",
    "JPM", "XOM", "JNJ", "WMT", "BRK.B",
]
# Short / common names that often collide in org-string matching
AMBIGUOUS_TICKERS = ["C", "T", "F", "GM", "META", "BRK.B", "TGT", "MA", "V", "DD"]

LOOKBACK_START = "2015-02-01"  # GKG 2.0 era
CLEAN_WINDOW_START = "2023-01-01"
CLEAN_WINDOW_END = "2023-03-31"
MAPPING_SAMPLE_START = "2024-01-01"
MAPPING_SAMPLE_END = "2024-01-31"
LIVE_N_FILES = 4  # recent 15-min GKG zips to pull in §3

# GKG 2.1 CSV is tab-delimited, no header (codebook column indices)
GKG_COL_DATE = 1
GKG_COL_SOURCE = 3
GKG_COL_V2ORGS = 14
GKG_COL_V2TONE = 15

LEGAL_SUFFIX_RE = re.compile(
    r"\b(inc|incorporated|corp|corporation|co|company|ltd|limited|llc|"
    r"plc|lp|l\.p\.|n\.v\.|sa|ag|nv|plc\.|group|holdings|holding)\b\.?",
    re.IGNORECASE,
)

print(f"ROOT={ROOT}")
print(f"TRAIN_PANEL exists={os.path.exists(TRAIN_PANEL)}")
print(f"CACHE_DIR={CACHE_DIR}")


In [ ]:
panel_meta = pd.read_parquet(TRAIN_PANEL, columns=["date", "ticker"])
panel_meta["date"] = pd.to_datetime(panel_meta["date"])
PANEL_TICKERS = sorted(panel_meta["ticker"].unique().tolist())
LOOKBACK_END = panel_meta["date"].max().strftime("%Y-%m-%d")

# Keep FOCUS / AMBIGUOUS lists to names present on the panel
FOCUS_TICKERS = [t for t in FOCUS_TICKERS if t in PANEL_TICKERS]
if "GOOGL" not in FOCUS_TICKERS and "GOOG" in PANEL_TICKERS:
    FOCUS_TICKERS.append("GOOG")
AMBIGUOUS_TICKERS = [t for t in AMBIGUOUS_TICKERS if t in PANEL_TICKERS]

print(
    f"panel tickers={len(PANEL_TICKERS)}  "
    f"date={panel_meta['date'].min().date()} -> {panel_meta['date'].max().date()}"
)
print(f"LOOKBACK_END={LOOKBACK_END}")
print(f"FOCUS_TICKERS={FOCUS_TICKERS}")
print(f"AMBIGUOUS_TICKERS={AMBIGUOUS_TICKERS}")
assert len(PANEL_TICKERS) > 0, "train panel returned no tickers"


In [ ]:
def _canonical_sec_ticker(symbol: str) -> str:
    return symbol.strip().upper().replace(".", "-")


def load_sec_ticker_map(cache_dir: str = CACHE_DIR) -> pd.DataFrame:
    """SEC company_tickers.json -> columns ticker, cik, title (project ticker form)."""
    path = os.path.join(cache_dir, "sec_company_tickers.json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
    else:
        resp = requests.get(
            SEC_TICKER_MAP_URL,
            headers={"User-Agent": SEC_USER_AGENT, "Accept-Encoding": "gzip, deflate"},
            timeout=60,
        )
        resp.raise_for_status()
        raw = resp.json()
        with open(path, "w", encoding="utf-8") as f:
            json.dump(raw, f)

    rows = []
    for entry in raw.values():
        sec_ticker = str(entry.get("ticker", "")).strip().upper()
        title = str(entry.get("title", "")).strip()
        cik = entry.get("cik_str")
        if not sec_ticker or not title or cik is None:
            continue
        # Project form: BRK-B -> BRK.B
        project_ticker = sec_ticker.replace("-", ".")
        rows.append(
            {
                "ticker": project_ticker,
                "sec_ticker": sec_ticker,
                "cik": str(int(cik)).zfill(10),
                "title": title,
            }
        )
    return pd.DataFrame(rows)


def strip_legal_suffixes(name: str) -> str:
    cleaned = LEGAL_SUFFIX_RE.sub("", name)
    cleaned = re.sub(r"[,.\s]+$", "", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()
    return cleaned


def build_aliases(title: str, extra: Iterable[str] | None = None) -> list[str]:
    """Build case-insensitive org aliases from an SEC title (+ optional extras)."""
    aliases: list[str] = []
    seen: set[str] = set()

    def _add(s: str) -> None:
        s = (s or "").strip()
        if len(s) < 2:
            return
        key = s.casefold()
        if key in seen:
            return
        seen.add(key)
        aliases.append(s)

    _add(title)
    stripped = strip_legal_suffixes(title)
    _add(stripped)
    # First token often matches GDELT short form ("Apple", "Microsoft")
    if stripped:
        first = stripped.split()[0]
        if len(first) >= 3 and first.casefold() not in {
            "the", "and", "for", "bank", "national", "united", "american"
        }:
            _add(first)
    if extra:
        for e in extra:
            _add(e)
    return aliases


# Manual extras for names that GDELT often labels differently
MANUAL_EXTRAS: dict[str, list[str]] = {
    "AAPL": ["Apple", "Apple Inc"],
    "MSFT": ["Microsoft", "Microsoft Corp"],
    "AMZN": ["Amazon", "Amazon.com", "Amazon Com"],
    "GOOGL": ["Google", "Alphabet", "Alphabet Inc"],
    "GOOG": ["Google", "Alphabet", "Alphabet Inc"],
    "META": ["Meta Platforms", "Facebook", "Meta"],
    "BRK.B": ["Berkshire Hathaway", "Berkshire"],
    "JPM": ["JPMorgan", "JP Morgan", "J.P. Morgan", "JPMorgan Chase"],
    "WMT": ["Walmart", "Wal-Mart"],
    "XOM": ["Exxon", "ExxonMobil", "Exxon Mobil"],
    "JNJ": ["Johnson & Johnson", "Johnson and Johnson"],
    "TGT": ["Target Corp", "Target Corporation"],
    "C": ["Citigroup", "Citibank", "Citi"],
    "T": ["AT&T", "AT and T"],
    "V": ["Visa Inc", "Visa"],
    "MA": ["Mastercard", "MasterCard"],
}


def parse_v2tone(raw) -> float | None:
    """First field of V2Tone CSV string -> overall document tone, else None."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    s = str(raw).strip()
    if not s:
        return None
    first = s.split(",")[0].strip()
    try:
        return float(first)
    except ValueError:
        return None


def match_orgs_to_ticker(v2orgs: str, alias_map: dict[str, list[str]]) -> list[str]:
    """Return tickers whose aliases appear as substrings in V2Organizations."""
    if not isinstance(v2orgs, str) or not v2orgs.strip():
        return []
    text = v2orgs.casefold()
    hits = []
    for ticker, aliases in alias_map.items():
        for alias in aliases:
            a = alias.casefold()
            if len(a) < 2:
                continue
            if a in text:
                hits.append(ticker)
                break
    return hits


def explode_org_matches(
    frame: pd.DataFrame,
    alias_map: dict[str, list[str]],
    *,
    orgs_col: str = "V2Organizations",
    tone_col: str = "V2Tone",
    date_col: str = "gkg_date",
) -> pd.DataFrame:
    """Parse tone, match aliases, explode to long (date, ticker, tone) rows."""
    if frame.empty:
        return pd.DataFrame(columns=["date", "ticker", "tone"])

    work = frame[[orgs_col, tone_col, date_col]].copy()
    work["tone"] = work[tone_col].map(parse_v2tone)
    work = work.dropna(subset=["tone"])
    work["tickers"] = work[orgs_col].fillna("").map(
        lambda s: match_orgs_to_ticker(s, alias_map)
    )
    work = work[work["tickers"].map(bool)]
    if work.empty:
        return pd.DataFrame(columns=["date", "ticker", "tone"])

    work = work.explode("tickers").rename(columns={"tickers": "ticker"})
    work["date"] = pd.to_datetime(
        work[date_col].astype(str).str.slice(0, 8),
        format="%Y%m%d",
        errors="coerce",
    )
    work = work.dropna(subset=["date"])
    return work[["date", "ticker", "tone"]].reset_index(drop=True)


def get_bq_client():
    if bigquery is None:
        raise ImportError("Install google-cloud-bigquery before running BigQuery cells")
    return bigquery.Client()


print("helpers ready")


In [ ]:
sec_map = load_sec_ticker_map()
sec_by_ticker = sec_map.drop_duplicates("ticker").set_index("ticker")

missing_sec = [t for t in PANEL_TICKERS if t not in sec_by_ticker.index]
print(f"SEC map rows={len(sec_map)}  panel missing from SEC={missing_sec}")

alias_map: dict[str, list[str]] = {}
alias_rows = []
for t in PANEL_TICKERS:
    if t in sec_by_ticker.index:
        title = str(sec_by_ticker.loc[t, "title"])
        cik = str(sec_by_ticker.loc[t, "cik"])
    else:
        title = t
        cik = ""
    aliases = build_aliases(title, extra=MANUAL_EXTRAS.get(t, []))
    alias_map[t] = aliases
    alias_rows.append(
        {
            "ticker": t,
            "cik": cik,
            "title": title,
            "n_aliases": len(aliases),
            "aliases": " | ".join(aliases),
        }
    )

alias_df = pd.DataFrame(alias_rows).sort_values("ticker")
alias_path = os.path.join(CACHE_DIR, "ticker_org_aliases.parquet")
alias_df.to_parquet(alias_path, index=False)
print(f"alias table cached -> {alias_path}")
alias_df.head(12)


## 1. Lookback depth & data cleanliness (Q1)

Uses BigQuery `gdelt-bq.gdeltv2.gkg_partitioned`. **Always** constrain `_PARTITIONTIME` so queries stay inside the free-tier scan budget.

1. Confirm GKG archive span near the focus names.
2. Per-year article counts for `FOCUS_TICKERS` from `LOOKBACK_START` through `LOOKBACK_END`.
3. Tone cleanliness on `CLEAN_WINDOW_*` (parse fail rate, histogram, sparsity, extremes).


In [ ]:
bq = get_bq_client()
print(f"BigQuery project={bq.project}")


def bq_query(sql: str) -> pd.DataFrame:
    job = bq.query(sql)
    return job.result().to_dataframe(create_bqstorage_client=False)


# 1.1 Archive existence - earliest / latest partition that mentions any focus alias
focus_aliases = sorted({a for t in FOCUS_TICKERS for a in alias_map[t]})
like_parts = []
for a in focus_aliases:
    esc = a.replace("\\", "\\\\").replace("'", "\\'")
    like_parts.append(f"LOWER(V2Organizations) LIKE '%{esc.casefold()}%'")
focus_like_sql = " OR ".join(like_parts)

sql_span = f"""
SELECT
  MIN(DATE(_PARTITIONTIME)) AS min_partition,
  MAX(DATE(_PARTITIONTIME)) AS max_partition,
  COUNT(*) AS n_rows
FROM {BQ_GKG_TABLE}
WHERE DATE(_PARTITIONTIME) BETWEEN DATE('{LOOKBACK_START}') AND DATE('{LOOKBACK_END}')
  AND V2Organizations IS NOT NULL
  AND ({focus_like_sql})
"""
span_df = bq_query(sql_span)
span_path = os.path.join(CACHE_DIR, "lookback_span.parquet")
span_df.to_parquet(span_path, index=False)
print(span_df.to_string(index=False))
print(f"cached -> {span_path}")


In [ ]:
# 1.2 Per-year coverage for focus tickers (one query per year to bound scan size)
year_rows = []
start_year = int(LOOKBACK_START[:4])
end_year = int(LOOKBACK_END[:4])

for year in range(start_year, end_year + 1):
    y0 = f"{year}-01-01"
    y1 = f"{year}-12-31"
    if year == start_year:
        y0 = LOOKBACK_START
    if year == end_year:
        y1 = LOOKBACK_END
    sql = f"""
    SELECT
      {year} AS year,
      COUNT(*) AS n_articles
    FROM {BQ_GKG_TABLE}
    WHERE DATE(_PARTITIONTIME) BETWEEN DATE('{y0}') AND DATE('{y1}')
      AND V2Organizations IS NOT NULL
      AND ({focus_like_sql})
    """
    part = bq_query(sql)
    year_rows.append(part.iloc[0].to_dict())
    print(f"year={year} n_articles={int(part.iloc[0]['n_articles'])}")

yearly = pd.DataFrame(year_rows)
yearly_path = os.path.join(CACHE_DIR, "lookback_yearly_focus.parquet")
yearly.to_parquet(yearly_path, index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(yearly["year"].astype(int), yearly["n_articles"])
ax.set_xlabel("year")
ax.set_ylabel("GKG rows matching any focus alias")
ax.set_title("Focus-universe GDELT coverage by year")
plt.tight_layout()
plt.show()
yearly


In [ ]:
# Per-ticker × year hit counts via SQL COUNT (avoids downloading full-year GKG rows)
def ticker_like_sql(ticker: str) -> str:
    parts = []
    for a in alias_map[ticker]:
        if len(a) < 3:
            continue
        esc = a.replace("\\", "\\\\").replace("'", "\\'")
        parts.append(f"LOWER(V2Organizations) LIKE '%{esc.casefold()}%'")
    if not parts:
        return "FALSE"
    return "(" + " OR ".join(parts) + ")"


per_ticker_year_rows = []
for year in range(start_year, end_year + 1):
    y0 = f"{year}-01-01"
    y1 = f"{year}-12-31"
    if year == start_year:
        y0 = LOOKBACK_START
    if year == end_year:
        y1 = LOOKBACK_END
    # One query returns a column per focus ticker
    select_bits = [
        f"COUNTIF({ticker_like_sql(t)}) AS `{t.replace('.', '_')}`" for t in FOCUS_TICKERS
    ]
    sql = f"""
    SELECT
      {", ".join(select_bits)}
    FROM {BQ_GKG_TABLE}
    WHERE DATE(_PARTITIONTIME) BETWEEN DATE('{y0}') AND DATE('{y1}')
      AND V2Organizations IS NOT NULL
      AND ({focus_like_sql})
    """
    part = bq_query(sql)
    row = part.iloc[0].to_dict()
    counts = {}
    for t in FOCUS_TICKERS:
        key = t.replace(".", "_")
        counts[t] = int(row.get(key, 0) or 0)
        per_ticker_year_rows.append({"year": year, "ticker": t, "n_articles": counts[t]})
    tickers_hit = sum(1 for n in counts.values() if n > 0)
    print(f"year={year} tickers_with_hit={tickers_hit}/{len(FOCUS_TICKERS)} counts={counts}")

ticker_year = pd.DataFrame(per_ticker_year_rows)
ticker_year_path = os.path.join(CACHE_DIR, "lookback_ticker_year.parquet")
ticker_year.to_parquet(ticker_year_path, index=False)

pivot = ticker_year.pivot(index="ticker", columns="year", values="n_articles").fillna(0).astype(int)
display(pivot)

fig, ax = plt.subplots(figsize=(9, 4))
hit_by_year = (
    ticker_year.assign(hit=lambda d: d["n_articles"] > 0)
    .groupby("year")["hit"]
    .sum()
)
ax.plot(hit_by_year.index, hit_by_year.values, marker="o")
ax.set_ylim(0, len(FOCUS_TICKERS) + 0.5)
ax.set_xlabel("year")
ax.set_ylabel("focus tickers with >=1 article")
ax.set_title("Focus ticker coverage breadth by year")
plt.tight_layout()
plt.show()


In [ ]:
# 1.3 Tone cleanliness - CLEAN_WINDOW on focus aliases
sql_clean = f'''
SELECT
  DATE AS gkg_date,
  V2Organizations,
  V2Tone,
  SourceCommonName
FROM {BQ_GKG_TABLE}
WHERE DATE(_PARTITIONTIME) BETWEEN DATE('{CLEAN_WINDOW_START}') AND DATE('{CLEAN_WINDOW_END}')
  AND V2Organizations IS NOT NULL
  AND ({focus_like_sql})
'''
clean_raw = bq_query(sql_clean)
print(f"cleanliness pull rows={len(clean_raw)}")

focus_alias_subset = {t: alias_map[t] for t in FOCUS_TICKERS}
n_raw = len(clean_raw)
n_null_tone = int(clean_raw["V2Tone"].isna().sum()) if n_raw else 0
parsed = clean_raw["V2Tone"].map(parse_v2tone)
n_parse_fail = int(((~clean_raw["V2Tone"].isna()) & parsed.isna()).sum()) if n_raw else 0

scored = explode_org_matches(clean_raw, focus_alias_subset)
scored_path = os.path.join(CACHE_DIR, "cleanliness_scored_focus.parquet")
scored.to_parquet(scored_path, index=False)

parse_fail_rate = n_parse_fail / n_raw if n_raw else float("nan")
null_tone_rate = n_null_tone / n_raw if n_raw else float("nan")
extreme_share = float((scored["tone"].abs() > 20).mean()) if len(scored) else float("nan")

print(f"raw_rows={n_raw}  scored_ticker_rows={len(scored)}")
print(f"null_tone_rate={null_tone_rate:.4%}  parse_fail_rate={parse_fail_rate:.4%}")
if len(scored):
    print(f"tone describe:\n{scored['tone'].describe()}")
print(f"share_|tone|>20={extreme_share:.4%}")

daily = (
    scored.groupby(["date", "ticker"], as_index=False)
    .agg(n_articles=("tone", "size"), mean_tone=("tone", "mean"))
    if len(scored)
    else pd.DataFrame(columns=["date", "ticker", "n_articles", "mean_tone"])
)
cal = pd.bdate_range(CLEAN_WINDOW_START, CLEAN_WINDOW_END)
grid = pd.MultiIndex.from_product([cal, FOCUS_TICKERS], names=["date", "ticker"]).to_frame(
    index=False
)
grid["date"] = pd.to_datetime(grid["date"])
merged = grid.merge(daily, on=["date", "ticker"], how="left")
merged["n_articles"] = merged["n_articles"].fillna(0).astype(int)
zero_share = float((merged["n_articles"] == 0).mean())
print(f"ticker-day zero-article share (business days)={zero_share:.2%}")
print(merged["n_articles"].describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
if len(scored):
    axes[0].hist(scored["tone"].dropna(), bins=50, color="steelblue", edgecolor="none")
axes[0].set_title("V2Tone (focus matches)")
axes[0].set_xlabel("tone")
pos = merged.loc[merged["n_articles"] > 0, "n_articles"]
if len(pos):
    axes[1].hist(pos, bins=40, color="darkorange")
axes[1].set_title("Articles per ticker-day (when >0)")
axes[1].set_xlabel("n_articles")
plt.tight_layout()
plt.show()

clean_summary = pd.DataFrame(
    [
        {
            "raw_rows": n_raw,
            "scored_rows": len(scored),
            "null_tone_rate": null_tone_rate,
            "parse_fail_rate": parse_fail_rate,
            "extreme_abs_gt_20": extreme_share,
            "zero_article_ticker_day_share": zero_share,
            "tone_mean": float(scored["tone"].mean()) if len(scored) else None,
            "tone_std": float(scored["tone"].std()) if len(scored) else None,
        }
    ]
)
clean_summary.to_parquet(os.path.join(CACHE_DIR, "cleanliness_summary.parquet"), index=False)
clean_summary


### 1.4 Lookback verdict (fill after running cells above)

After executing §1, record:

- Earliest year with non-trivial focus coverage.
- Whether daily features look viable from 2015, or only from a later year (e.g. 2018/2020).
- Cleanliness flags (null/parse rates, sparsity, extreme tones) that would push you toward FinBERT later.

The §4 decision cell prints a machine-readable summary from cached parquets.


## 2. Ticker → GDELT organization mapping (Q2)

Map every panel ticker via SEC `title` (+ manual extras) to substrings in `V2Organizations`. Validate on one recent partition month (`MAPPING_SAMPLE_*`) for **all** panel tickers; audit ambiguous names separately.


In [ ]:
# Build LIKE clause for all panel aliases (one sample month)
all_aliases = sorted({a for t in PANEL_TICKERS for a in alias_map[t]})
print(f"panel aliases={len(all_aliases)}")

like_all = []
for a in all_aliases:
    esc = a.replace("\\", "\\\\").replace("'", "\\'")
    # skip very short aliases in SQL filter to reduce false positives / scan noise
    if len(a) < 3:
        continue
    like_all.append(f"LOWER(V2Organizations) LIKE '%{esc.casefold()}%'")
panel_like_sql = " OR ".join(like_all)

sql_map = f"""
SELECT
  V2Organizations,
  V2Tone
FROM {BQ_GKG_TABLE}
WHERE DATE(_PARTITIONTIME) BETWEEN DATE('{MAPPING_SAMPLE_START}') AND DATE('{MAPPING_SAMPLE_END}')
  AND V2Organizations IS NOT NULL
  AND ({panel_like_sql})
"""
map_raw = bq_query(sql_map)
print(f"mapping sample rows={len(map_raw)}")

hit_counts = {t: 0 for t in PANEL_TICKERS}
example_orgs: dict[str, list[str]] = {t: [] for t in PANEL_TICKERS}
org_string_counts: dict[str, dict[str, int]] = {t: defaultdict(int) for t in PANEL_TICKERS}

for orgs in map_raw["V2Organizations"].fillna(""):
    matched = match_orgs_to_ticker(orgs, alias_map)
    if not matched:
        continue
    tokens = [tok.strip() for tok in re.split(r"[;,]", orgs) if tok.strip()]
    for t in matched:
        hit_counts[t] += 1
        for tok in tokens:
            if any(a.casefold() in tok.casefold() for a in alias_map[t]):
                org_string_counts[t][tok] += 1
                if len(example_orgs[t]) < 5 and tok not in example_orgs[t]:
                    example_orgs[t].append(tok)

qa_rows = []
for t in PANEL_TICKERS:
    n = hit_counts[t]
    title = alias_df.loc[alias_df["ticker"] == t, "title"]
    title_s = title.iloc[0] if len(title) else ""
    if t in AMBIGUOUS_TICKERS:
        status = "ambiguous"
    elif n == 0:
        status = "sparse"
    elif n < 5:
        status = "sparse"
    else:
        status = "ok"
    qa_rows.append(
        {
            "ticker": t,
            "title": title_s,
            "n_articles": n,
            "example_orgs": " || ".join(example_orgs[t]),
            "status": status,
        }
    )

mapping_qa = pd.DataFrame(qa_rows).sort_values(["status", "n_articles", "ticker"])
mapping_path = os.path.join(CACHE_DIR, "mapping_qa.parquet")
mapping_qa.to_parquet(mapping_path, index=False)

status_counts = mapping_qa["status"].value_counts()
print(status_counts.to_string())
print(f"hit rate (>=1 article)={(mapping_qa['n_articles'] > 0).mean():.1%}")
print(f"cached -> {mapping_path}")
mapping_qa.head(20)


In [ ]:
# Ambiguity audit - top matched V2Organizations tokens for watchlist names
amb_rows = []
for t in AMBIGUOUS_TICKERS:
    counts = org_string_counts.get(t, {})
    top = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:15]
    title_s = (
        alias_df.loc[alias_df["ticker"] == t, "title"].iloc[0]
        if (alias_df["ticker"] == t).any()
        else "?"
    )
    for org, n in top:
        amb_rows.append({"ticker": t, "org_string": org, "n": n})
    print(f"\n=== {t}  title={title_s}  hits={hit_counts[t]} ===")
    for org, n in top:
        print(f"  {n:5d}  {org}")

amb_df = pd.DataFrame(amb_rows)
amb_path = os.path.join(CACHE_DIR, "mapping_ambiguity.parquet")
amb_df.to_parquet(amb_path, index=False)

print(
    "\nInterpretation guide: if top org strings are clearly the listed company, "
    "status 'ambiguous' is caution-only. If they are unrelated entities sharing a "
    "short token (e.g. generic 'Target', single-letter banks), add stricter aliases "
    "before production."
)
amb_df.head(30)


### 2.3 Mapping conclusion (fill after running)

- Viable as-is if most liquid names are `ok` and ambiguous audits look company-specific.
- Needs manual aliases if several large-cap names are `sparse`.
- Blocking if short tickers systematically match unrelated orgs with no fix via longer aliases.


## 3. Daily live access pattern (Q3)

No BigQuery. Mirrors a future `fetch_gdelt_tone_daily`:

1. Read [`lastupdate.txt`](http://data.gdeltproject.org/gdeltv2/lastupdate.txt) for the newest GKG zip URL(s).
2. Download a small number of recent 15-minute GKG files (`LIVE_N_FILES`).
3. Filter with the **same** `alias_map` as §2; aggregate to `date, ticker, n_articles, mean_tone, median_tone`.

**Production cadence (for later pipeline):** after the US cash close, download all GKG zips whose timestamps fall in the trading-day window (or prior 24h UTC), aggregate with this entity map, write parquet under `01_data/cache/`, then `merge_asof(..., direction='backward')` onto the OHLCV panel so bar `t` only sees tone known by decision time `t`.


In [ ]:
def fetch_lastupdate_lines(url: str = GDELT_LASTUPDATE_URL) -> list[str]:
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return [ln.strip() for ln in resp.text.splitlines() if ln.strip()]


def latest_gkg_stamp(lines: list[str]) -> str:
    """Return YYYYMMDDHHMMSS stamp from the newest gkg.csv.zip line in lastupdate."""
    for ln in lines:
        parts = ln.split()
        if len(parts) < 3:
            continue
        url = parts[-1]
        if "gkg.csv.zip" not in url.lower():
            continue
        base = os.path.basename(url)
        # e.g. 20240722121500.gkg.csv.zip
        stamp = base.split(".")[0]
        if len(stamp) == 14 and stamp.isdigit():
            return stamp
    raise ValueError("No gkg.csv.zip stamp found in lastupdate.txt")


def gkg_urls_walking_back(stamp: str, n: int = LIVE_N_FILES) -> list[str]:
    """Build n GKG zip URLs at 15-minute steps ending at stamp (inclusive)."""
    ts = pd.to_datetime(stamp, format="%Y%m%d%H%M%S")
    urls = []
    for i in range(n):
        t = ts - pd.Timedelta(minutes=15 * i)
        s = t.strftime("%Y%m%d%H%M%S")
        urls.append(f"http://data.gdeltproject.org/gdeltv2/{s}.gkg.csv.zip")
    return urls


def load_gkg_zip(url: str, cache_dir: str = CACHE_DIR) -> pd.DataFrame | None:
    fname = os.path.basename(url)
    local = os.path.join(cache_dir, fname)
    if not os.path.exists(local):
        print(f"downloading {url}")
        resp = requests.get(url, timeout=180)
        if resp.status_code == 404:
            print(f"  skip 404 {url}")
            return None
        resp.raise_for_status()
        with open(local, "wb") as f:
            f.write(resp.content)
    else:
        print(f"cache hit {local}")

    with zipfile.ZipFile(local) as zf:
        names = zf.namelist()
        assert names, f"empty zip: {local}"
        with zf.open(names[0]) as fh:
            df = pd.read_csv(
                fh,
                sep="\t",
                header=None,
                dtype=str,
                usecols=[GKG_COL_DATE, GKG_COL_SOURCE, GKG_COL_V2ORGS, GKG_COL_V2TONE],
                names=["gkg_date", "SourceCommonName", "V2Organizations", "V2Tone"],
                na_filter=False,
            )
    return df


lines = fetch_lastupdate_lines()
stamp = latest_gkg_stamp(lines)
gkg_urls = gkg_urls_walking_back(stamp, n=LIVE_N_FILES)
print("lastupdate (first 5 lines):")
for ln in lines[:5]:
    print(" ", ln)
print(f"\nlatest stamp={stamp}")
print(f"GKG urls to pull ({len(gkg_urls)}):")
for u in gkg_urls:
    print(" ", u)


In [ ]:
frames = []
for u in gkg_urls:
    df = load_gkg_zip(u)
    if df is not None:
        frames.append(df)
assert frames, "No GKG zips downloaded - check network / stamps"
live_raw = pd.concat(frames, ignore_index=True)
print(f"live raw rows={len(live_raw)} files={len(frames)}")

live_scored = explode_org_matches(live_raw, alias_map)
if live_scored.empty:
    live_daily = pd.DataFrame(
        columns=["date", "ticker", "n_articles", "mean_tone", "median_tone"]
    )
    print(
        "WARNING: no panel tickers matched in the recent GKG window - "
        "widen LIVE_N_FILES or check aliases"
    )
else:
    live_daily = (
        live_scored.groupby(["date", "ticker"], as_index=False)
        .agg(
            n_articles=("tone", "size"),
            mean_tone=("tone", "mean"),
            median_tone=("tone", "median"),
        )
        .sort_values(["date", "ticker"])
    )

live_path = os.path.join(CACHE_DIR, "live_daily_tone.parquet")
live_daily.to_parquet(live_path, index=False)
print(f"live daily shape={live_daily.shape}  cached -> {live_path}")
print(f"tickers hit={live_daily['ticker'].nunique() if len(live_daily) else 0}")
print(
    f"focus tickers hit="
    f"{sorted(set(live_daily['ticker']).intersection(FOCUS_TICKERS)) if len(live_daily) else []}"
)

expected_cols = {"date", "ticker", "n_articles", "mean_tone", "median_tone"}
assert live_daily.empty or expected_cols <= set(live_daily.columns)
if len(live_daily):
    assert live_daily["n_articles"].min() >= 1
    n_focus = int(live_daily["ticker"].isin(FOCUS_TICKERS).sum())
    print(f"focus ticker-day rows={n_focus}")
else:
    print("empty live_daily - check network / aliases / file count")

live_daily.head(20)


### 3.3 Live-path notes

- Each 15-minute GKG zip is large; **always** filter to your alias set in-process (or push filters into a future fetcher) — do not retain full worldwide GKG forever.
- Train history (BigQuery) and live (HTTP) must share one frozen `alias_map` artifact (this notebook writes `ticker_org_aliases.parquet`).
- For PIT joins onto the equity panel: `merge_asof(..., direction='backward')` on `date` within each ticker after aggregating tone to the trading date you will decide on.


## 4. Decision summary

Answers the three probe questions from cached outputs. Re-run after §1–§3. This cell does **not** implement a production fetcher — it only records go/no-go for that next step.


In [ ]:
def _safe_read(name: str) -> pd.DataFrame | None:
    path = os.path.join(CACHE_DIR, name)
    if not os.path.exists(path):
        print(f"missing cache: {path}")
        return None
    return pd.read_parquet(path)


yearly_c = _safe_read("lookback_yearly_focus.parquet")
ticker_year_c = _safe_read("lookback_ticker_year.parquet")
clean_c = _safe_read("cleanliness_summary.parquet")
mapping_c = _safe_read("mapping_qa.parquet")
live_c = _safe_read("live_daily_tone.parquet")

print("=" * 72)
print("Q1 - How far does the data go back?")
if yearly_c is not None and len(yearly_c):
    nonzero = yearly_c.loc[yearly_c["n_articles"] > 0, "year"]
    earliest = int(nonzero.min()) if len(nonzero) else None
    latest = int(nonzero.max()) if len(nonzero) else None
    print(f"  Focus-alias article mass present from year={earliest} -> {latest}")
    print(f"  Yearly counts:\n{yearly_c.to_string(index=False)}")
else:
    print("  (run §1 yearly cell)")

if ticker_year_c is not None and len(ticker_year_c):
    breadth = (
        ticker_year_c.assign(hit=lambda d: d["n_articles"] > 0)
        .groupby("year")["hit"]
        .sum()
    )
    print(f"  Focus tickers with >=1 hit by year:\n{breadth.to_string()}")
    usable_years = breadth[breadth >= max(1, int(0.7 * breadth.max()))].index.tolist()
    print(f"  Heuristic continuous-breadth years (>=70% of peak): {usable_years}")

if clean_c is not None and len(clean_c):
    print("  Cleanliness snapshot:")
    print(clean_c.to_string(index=False))

print()
print("Q2 - Ticker -> org mapping - will it be a problem?")
if mapping_c is not None and len(mapping_c):
    vc = mapping_c["status"].value_counts()
    hit_rate = float((mapping_c["n_articles"] > 0).mean())
    print(f"  Sample window hit rate={hit_rate:.1%}")
    print(f"  Status counts:\n{vc.to_string()}")
    problem = mapping_c.loc[
        mapping_c["status"] != "ok", ["ticker", "title", "n_articles", "status"]
    ]
    print(f"  Non-ok tickers ({len(problem)}):")
    print(problem.head(30).to_string(index=False))
else:
    print("  (run §2 mapping cell)")

print()
print("Q3 - Daily live access?")
if live_c is not None:
    print(
        f"  live_daily rows={len(live_c)}  "
        f"tickers={live_c['ticker'].nunique() if len(live_c) else 0}  "
        f"cols={list(live_c.columns)}"
    )
    print(
        "  Pattern validated: lastupdate.txt -> gkg.csv.zip -> alias filter -> "
        "daily mean/median tone. Same alias_map as history."
    )
else:
    print("  (run §3 live cells)")

print()
print("GO / NO-GO for production GDELT fetcher")
go = True
reasons = []
if yearly_c is None or mapping_c is None or live_c is None:
    go = False
    reasons.append("incomplete probe caches - finish §1–§3")
else:
    if (mapping_c["n_articles"] > 0).mean() < 0.5:
        go = False
        reasons.append("mapping hit rate < 50% on sample month")
    if len(live_c) == 0:
        go = False
        reasons.append("live aggregation returned 0 rows")
    if yearly_c["n_articles"].sum() == 0:
        go = False
        reasons.append("no historical focus articles found")
    if go:
        reasons.append(
            "history + mapping + live path look workable - next: "
            "01_data/ingestion/alternative_data/gdelt_fetcher.py "
            "(still out of scope for this notebook)"
        )

print(f"  decision={'GO' if go else 'NO-GO'}")
for r in reasons:
    print(f"  - {r}")
print("=" * 72)
